In [1]:
import os

In [3]:
%pwd

'c:\\Users\\abhin\\Desktop\\Mlops\\Deep-Learning-Kidney-Tumor-Classification-\\research'

In [4]:
os.chdir("../")

In [5]:
%pwd

'c:\\Users\\abhin\\Desktop\\Mlops\\Deep-Learning-Kidney-Tumor-Classification-'

In [26]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class TrainingPipelineConfig:
    root_dir: Path
    trained_model_path: Path
    updated_base_model_path: Path
    trained_data_path: Path
    params_epochs: int
    params_batch_size: int
    params_is_use_augmentation:bool 
    params_image_size:list
  

In [27]:
from cnnclassifier.constants import *
from cnnclassifier.utils.common import read_yaml, create_directories

In [28]:
import tensorflow as tf

In [33]:
class configuartionManager:
    def __init__(self, config_file_path = config_file_path, params_file_path = params_file_path):
        self.config = read_yaml(config_file_path)
        self.params = read_yaml(params_file_path)
        create_directories([self.config.artifacts_root])

    def get_training_pipeline_config(self) -> TrainingPipelineConfig:
        training_config = self.config.training
        prepare_base_model= self.config.prepare_base_model
        params = self.params
        training_data = os.path.join(self.config.data_ingestion.unzip_dir,"KidneyData")

        
        training_pipeline_config = TrainingPipelineConfig(
            root_dir=Path(training_config.root_dir),
            trained_model_path=Path(training_config.trained_model_path),
            updated_base_model_path=Path(prepare_base_model.updated_base_model_path),
            trained_data_path=Path(training_data),
            params_epochs=params.EPOCHS,
            params_batch_size=params.BATCH_SIZE,
            params_is_use_augmentation=params.AUGMENTATION,
            params_image_size=params.IMAGE_SIZE
        )
        return training_pipeline_config

In [34]:
import os
import urllib.request as request
from zipfile import ZipFile
import tensorflow as tf
import time

In [37]:
class Training:
    def __init__(self, config: TrainingPipelineConfig):
        self.config = config

    def get_base_model(self):
        self.model = tf.keras.models.load_model(self.config.updated_base_model_path)

    def train_valid_generator(self):
        datagenerator_kwargs = dict(rescale=1./255, validation_split=0.20)  
        dataflow_kwargs = dict(target_size=self.config.params_image_size[:-1], batch_size=self.config.params_batch_size, interpolation="bilinear")
        valid_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(**datagenerator_kwargs)
        self.valid_generator = valid_datagenerator.flow_from_directory(directory=self.config.trained_data_path, subset="validation", shuffle=False, **dataflow_kwargs)
        if self.config.params_is_use_augmentation:
            train_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
                rotation_range=40,
                horizontal_flip=True,
                width_shift_range=0.2,
                height_shift_range=0.2,
                shear_range=0.2,
                zoom_range=0.2,
                **datagenerator_kwargs
            )
        else:
            train_datagenerator = valid_datagenerator

        self.train_generator = train_datagenerator.flow_from_directory(directory=self.config.trained_data_path, subset="training", shuffle=True, **dataflow_kwargs)

    @staticmethod
    def save_model(path: Path, model: tf.keras.Model):
        model.save(path)

    def train(self):
        self.steps_per_epoch = self.train_generator.samples // self.train_generator.batch_size
        self.validation_steps = self.valid_generator.samples // self.valid_generator.batch_size

        self.model.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate=0.01),
            loss="categorical_crossentropy",
            metrics=["accuracy"]
        )

        self.model.fit(
            self.train_generator,
            epochs=self.config.params_epochs,
            steps_per_epoch=self.steps_per_epoch,
            validation_steps=self.validation_steps,
            validation_data=self.valid_generator
        )

        self.save_model(
            path=self.config.trained_model_path,
            model=self.model
        )

In [38]:
import traceback

try:
    training_pipeline_config = configuartionManager().get_training_pipeline_config()
    training = Training(config=training_pipeline_config)
    training.get_base_model()
    training.train_valid_generator()
    training.train()

except Exception as e:
    print(f"Error: {e}")
    traceback.print_exc()

2026-08-04 14:48:37,742 - cnnclassifier - INFO - yaml file: config\config.yaml loaded successfully
2026-08-04 14:48:37,744 - cnnclassifier - INFO - yaml file: params.yaml loaded successfully
2026-08-04 14:48:37,746 - cnnclassifier - INFO - created directory at: artifacts
2026-08-04 14:48:37,949 - absl - WARNING - Compiled the loaded model, but the compiled metrics have yet to be built. `model.compile_metrics` will be empty until you train or evaluate the model.
Found 1471 images belonging to 2 classes.
Found 5889 images belonging to 2 classes.
368/368 ━━━━━━━━━━━━━━━━━━━━ 1251s 3s/step - accuracy: 0.7856 - loss: 2.4537 - val_accuracy: 0.7301 - val_loss: 5.8033
2026-08-04 15:09:29,874 - absl - WARNING - You are saving your model as an HDF5 file via `model.save()` or `keras.saving.save_model(model)`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')` or `keras.saving.save_model(model, 'my_model.keras')`. 
